# Cat vs Dog Classification with Transfer Learning
This notebook implements the end-to-end Deep Learning workflow for classifying Cat and Dog images as shown in the project reference.

## 1. Importing Libraries

In [ ]:
import os
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms, models

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. The Dataset into Training and Testing sets

In [ ]:
DATA_DIR = "archive/animals"
cat_dir = os.path.join(DATA_DIR, "cat")
dog_dir = os.path.join(DATA_DIR, "dog")

cat_images = len(os.listdir(cat_dir)) if os.path.exists(cat_dir) else 0
dog_images = len(os.listdir(dog_dir)) if os.path.exists(dog_dir) else 0

print(f"Cat images: {cat_images}")
print(f"Dog images: {dog_images}")
print(f"Total images: {cat_images + dog_images}")

## 3. Loading and Preprocessing Data

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Data Augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Preprocessing for validation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

base_dataset = datasets.ImageFolder(root=DATA_DIR)
classes = base_dataset.classes
total_size = len(base_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

train_subset, val_subset = random_split(
    base_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

class CustomDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_loader = DataLoader(CustomDataset(train_subset, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(CustomDataset(val_subset, val_transform), batch_size=32, shuffle=False)

print(f"Training batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

## 4. Build the Model (Transfer Learning with VGG16 / MobileNet)

In [ ]:
# Using MobileNetV2 / VGG16 feature extractor
base_model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze base convolutional layers
for param in base_model.features.parameters():
    param.requires_grad = False

# Custom classification head matching the notebook architecture:
# Flatten -> Dense(256, relu) -> Dropout(0.5) -> Dense(num_classes)
base_model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(base_model.last_channel, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5),
    nn.Linear(256, len(classes))
)

model = base_model.to(device)
print(model)

## 5. Compile the Model

In [ ]:
criterion = nn.CrossEntropyLoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=0.001)

## 6. Train the Model with Early Stopping

In [ ]:
epochs = 6
patience = 3
best_val_loss = float("inf")
patience_counter = 0
best_weights = None

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == lbls).sum().item()
        total_train += lbls.size(0)

    tr_loss = running_loss / total_train
    tr_acc = correct_train / total_train

    # Validation
    model.eval()
    v_loss, correct_v, total_v = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            v_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_v += (preds == lbls).sum().item()
            total_v += lbls.size(0)

    val_l = v_loss / total_v
    val_a = correct_v / total_v

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_l)
    history["val_acc"].append(val_a)

    print(f"Epoch {epoch:02d}/{epochs:02d} - loss: {tr_loss:.4f} - accuracy: {tr_acc:.4f} - val_loss: {val_l:.4f} - val_accuracy: {val_a:.4f}")

    # Early Stopping check
    if val_l < best_val_loss:
        best_val_loss = val_l
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}!")
            break

if best_weights is not None:
    model.load_state_dict(best_weights)

## 7. Printing Final Training Accuracy and Validation Accuracy

In [ ]:
print(f"Final Training Accuracy:   {history['train_acc'][-1] * 100:.2f}%")
print(f"Final Validation Accuracy: {history['val_acc'][-1] * 100:.2f}%")

## 8. Visualize the Results

In [ ]:
ep_range = range(1, len(history["train_acc"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep_range, history["train_acc"], label="Training Accuracy", color="#1f77b4")
ax1.plot(ep_range, history["val_acc"], label="Validation Accuracy", color="#ff7f0e")
ax1.set_title("Training and Validation Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend(loc="lower right")
ax1.grid(True)

ax2.plot(ep_range, history["train_loss"], label="Training Loss", color="#1f77b4")
ax2.plot(ep_range, history["val_loss"], label="Validation Loss", color="#ff7f0e")
ax2.set_title("Training and Validation Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend(loc="upper right")
ax2.grid(True)

plt.tight_layout()
plt.show()